# 第 12 章：MoE 融合算子和性能分析 — 章节介绍

## 1. 前置要求

学习本章节之前，请确保你已具备以下能力或已完成相关学习：

- **前置课程**：已完成第 2 章（栈的表达式求值：UB 普通数组模拟数据结构、纯标量算子范式）与第 8 章（工程部署及性能分析：算子工程全流程）。
- **Ascend C 基础**：了解自定义算子工程结构（op_host / op_kernel）、Tiling 的作用、aclnn 单算子调用方式。
- **环境**：具备可用的 CANN 9.0.0 + Atlas A2（Ascend 910B3）环境，或 CANNLab 云开发环境（镜像模板 `cann_9.0.0_py3.11-A2-arm`）。

> 若对第 8 章的算子工程全流程（编译 → 打包 → 部署 → aclnn 调用）不熟悉，建议先完成第 8 章再继续。

## 2. 章节目标

本实验以 **MoE（Mixture of Experts）Router 路径**（`matmul → softmax → topk → renorm`）为载体，
设计并实现一个 **4 合 1 融合算子**，并从**数据结构视角**完成性能分析：

1. 理解算子融合的可行性判据：子图中间张量容量 ≤ UB 且无跨核依赖；
2. 掌握中间张量的"介质降级"（GM → UB）与 HBM 流量估算法；
3. 理解 Top-K 是**选择问题**（selection）而非排序问题（sorting）；
4. 完成纯标量融合算子的工程全流程：编译 → 打包 → 部署 → aclnn 调用 → 8 组边界用例回归；
5. 掌握多核切分的硬件约束：标量 GM 写经 L2（line=64B），多核共写同一 line 丢写，
   需用连续块 + 64B 对齐切分规避；
6. 学会对融合算子做定量性能归因：访存收益（融合带来）与计算吞吐瓶颈（执行管线决定）。

## 3. 学习路径与内容导航

<table style="text-align: left; margin-left: 0;">
  <thead>
    <tr>
      <th>Notebook</th>
      <th>内容</th>
      <th>预计耗时</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><a href="12.01_chapter_intro.ipynb">12.01 章节介绍</a></td>
      <td>前置要求 / 目标 / 导航 / 融合数据流</td>
      <td>5 分钟</td>
    </tr>
    <tr>
      <td><a href="12.02_moe_router_fused_lab.ipynb">12.02 动手实验</a></td>
      <td>全流程 7 步骤（环境 → 基线 → 设计 → 源码 → 部署 → 回归 → 性能）</td>
      <td>60-90 分钟</td>
    </tr>
    <tr>
      <td><a href="12.03_chapter_test.ipynb">12.03 章节实践</a></td>
      <td>编程实践（简单/中等/困难）+ 知识测验</td>
      <td>30-60 分钟</td>
    </tr>
  </tbody>
</table>

## 4. MoE Router 计算链

MoE 模型中，**Router（门控网络）**为每个 token 从 E 个专家中选出 Top-K 个，并给出归一化权重：

```text
scores       = x @ W_gate                       # [N,D] × [D,E] → [N,E]，Cube/矩阵乘
gate_scores  = softmax(scores, dim=-1)          # 逐行归一，[N,E]
(topk_scores, topk_idx) = topk(gate_scores, K)  # 每行取最大的 K 个，[N,K]
topk_weights = topk_scores / Σ topk_scores      # renorm，行内和为 1，[N,K]
```

四个算子构成**链式 DAG**；三个中间张量 `scores`、`gate_scores`、`topk_scores`
形状分别为 [N,E]、[N,E]、[N,K]——**E ≤ 64 时规模很小**，是典型的融合对象。

## 5. 融合前后数据流对比

<div style="text-align: left;">
  <img src="./images/moe_dataflow.svg" alt="MoE Router 融合前后数据流对比" width="780">
</div>

未融合：3 个中间张量共 **16·N·E + 8·N·K 字节**的 GM 往返 + 4 次发射；
融合：中间张量全部驻留 UB（< 0.5KB/行），1 次发射。

## 6. 本章的数据结构视角

- **融合 = 子图压缩**：中间边的介质从 GM（慢、容量大）降级为 UB（快、192KB/核）。
  可行性判据：中间张量按行分块后容量 ≤ UB，且切分维度上无跨核依赖
  （反例：E 极大或 softmax 需全局归一时，融合收益消失）。
- **Top-K 是选择问题**：K 轮 "取最大 + 掩蔽"（O(K·E)）即可，无需全排序（O(E·log E)）——
  选择正确的数据结构/算法族，比优化常数更重要。
- **多核切分也是数据结构问题**：输出布局的行粒度（K·(4+2) 字节/行）与并发写单元粒度
  （64B cache line）的相对关系，决定了切分方案（跨步行进会让多核共写同一 line）。

**下一步**：打开 [12.02_moe_router_fused_lab.ipynb](12.02_moe_router_fused_lab.ipynb) 开始动手实验。
